# Inference

## DailyDialog

In [1]:
import re
import ast
import pandas as pd

df = pd.read_csv("DailyDialogue/test.csv")

In [2]:
def parse_dialog_text(x):
    """
    Handles:
    - already-a-list
    - stringified Python list, e.g. "['hi', 'hello']"
    - fallback to regex only if needed
    """
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return None

    if isinstance(x, str):
        x = x.strip()
           
        dialog = re.findall(r"'(.*?)'", x, flags=re.DOTALL)
        return dialog if dialog else None

    return None

def parse_label_list(x):
    if isinstance(x, list):
        return [int(v) for v in x]

    if pd.isna(x):
        return None

    if not isinstance(x, str):
        return None

    x = x.strip()

    try:
        val = ast.literal_eval(x)
        if isinstance(val, list):
            return [int(v) for v in val]
    except Exception:
        pass

    nums = re.findall(r"\d+", x)
    if nums:
        return [int(v) for v in nums]

    return None

### playground

In [3]:
import re
import ast
import pandas as pd
from typing import List, Optional, Dict, Any


# =========================================================
# Parsing functions
# Reuse / keep these
# ========================================================


# =========================================================
# Label mapping
# =========================================================
LABEL2NAME = {
    0: "no emotion",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise",
}


# =========================================================
# Helper: assign speakers by ABAB scheme
# =========================================================
def speaker_of_turn(turn_idx: int) -> str:
    return "A" if turn_idx % 2 == 0 else "B"


# =========================================================
# Core logic: find label shifts for one speaker
# =========================================================
def find_speaker_label_shifts(
    df: pd.DataFrame,
    dialog_col="dialog",
    emotion_col="emotion",
    require_nonzero=True,
    direct_consecutive_for_speaker=True,
    min_turn_gap_same_speaker=1,
):
    """
    Find places where the same speaker's label changes across their own turns.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain dialog and emotion columns.
    dialog_col : str
        Column name for dialogue list / stringified dialogue.
    emotion_col : str
        Column name for emotion label list / stringified labels.
    require_nonzero : bool
        If True, only count shifts where both old and new labels are non-zero.
    direct_consecutive_for_speaker : bool
        If True, only compare each speaker turn with that speaker's immediately previous turn.
        Under ABAB, speaker A's timeline is turn 0 -> 2 -> 4 -> ...
    min_turn_gap_same_speaker : int
        Minimum number of same-speaker occurrences between comparisons.
        Usually keep at 1.

    Returns
    -------
    results_df : pd.DataFrame
        One row per detected shift.
    """
    results = []

    for row_idx, row in enumerate(df.itertuples(index=False)):
        try:
            dialog_raw = getattr(row, dialog_col) if hasattr(row, dialog_col) else row[df.columns.get_loc(dialog_col)]
            emotion_raw = getattr(row, emotion_col) if hasattr(row, emotion_col) else row[df.columns.get_loc(emotion_col)]
        except Exception:
            dialog_raw = row[df.columns.get_loc(dialog_col)]
            emotion_raw = row[df.columns.get_loc(emotion_col)]

        dialog = parse_dialog_text(dialog_raw)
        emotions = parse_label_list(emotion_raw)

        if dialog is None or emotions is None:
            continue

        if len(dialog) != len(emotions):
            continue

        # collect each speaker's own timeline
        speaker_turns = {"A": [], "B": []}
        for i, (utt, lab) in enumerate(zip(dialog, emotions)):
            speaker = speaker_of_turn(i)
            speaker_turns[speaker].append({
                "global_turn_idx": i,
                "speaker_turn_idx": len(speaker_turns[speaker]),
                "utterance": utt,
                "label": int(lab),
                "label_name": LABEL2NAME.get(int(lab), str(lab)),
            })

        # inspect shifts within each speaker's own sequence
        for speaker in ["A", "B"]:
            seq = speaker_turns[speaker]
            if len(seq) < 2:
                continue

            if direct_consecutive_for_speaker:
                compare_pairs = [(i - 1, i) for i in range(1, len(seq))]
            else:
                compare_pairs = []
                for j in range(1, len(seq)):
                    for i in range(max(0, j - min_turn_gap_same_speaker), j):
                        compare_pairs.append((i, j))

            for i_prev, i_curr in compare_pairs:
                prev_item = seq[i_prev]
                curr_item = seq[i_curr]

                old_label = prev_item["label"]
                new_label = curr_item["label"]

                if old_label == new_label:
                    continue

                if require_nonzero and (old_label == 0 or new_label == 0):
                    continue

                results.append({
                    "dialogue_row_idx": row_idx,
                    "speaker": speaker,

                    "prev_global_turn_idx": prev_item["global_turn_idx"],
                    "curr_global_turn_idx": curr_item["global_turn_idx"],

                    "prev_speaker_turn_idx": prev_item["speaker_turn_idx"],
                    "curr_speaker_turn_idx": curr_item["speaker_turn_idx"],

                    "old_label": old_label,
                    "old_label_name": prev_item["label_name"],
                    "new_label": new_label,
                    "new_label_name": curr_item["label_name"],

                    "prev_utterance": prev_item["utterance"],
                    "curr_utterance": curr_item["utterance"],

                    "dialog": dialog,
                    "emotions": emotions,
                })

    results_df = pd.DataFrame(results)
    return results_df


# =========================================================
# Helper: filter for specific shifts
# =========================================================
def filter_specific_shift(
    shifts_df: pd.DataFrame,
    from_label: Optional[int] = None,
    to_label: Optional[int] = None,
    speaker: Optional[str] = None,
):
    out = shifts_df.copy()

    if from_label is not None:
        out = out[out["old_label"] == from_label]

    if to_label is not None:
        out = out[out["new_label"] == to_label]

    if speaker is not None:
        out = out[out["speaker"] == speaker]

    return out.reset_index(drop=True)


# =========================================================
# Pretty print matched cases
# =========================================================
def print_shift_examples(shifts_df: pd.DataFrame, n: int = 5):
    if len(shifts_df) == 0:
        print("No matches found.")
        return

    sample_df = shifts_df.head(n)

    for k, row in sample_df.iterrows():
        print("=" * 100)
        print(f"Match #{k}")
        print(f"Dialogue row idx: {row['dialogue_row_idx']}")
        print(f"Speaker: {row['speaker']}")
        print(
            f"Shift: {row['old_label']} ({row['old_label_name']})"
            f" -> {row['new_label']} ({row['new_label_name']})"
        )
        print(f"Previous turn idx (global): {row['prev_global_turn_idx']}")
        print(f"Current  turn idx (global): {row['curr_global_turn_idx']}")
        print()
        print("[Previous utterance]")
        print(row["prev_utterance"])
        print()
        print("[Current utterance]")
        print(row["curr_utterance"])
        print()
        print("[Dialogue excerpt up to current turn]")
        dialog = row["dialog"]
        curr_turn = row["curr_global_turn_idx"]
        for i in range(curr_turn + 1):
            spk = speaker_of_turn(i)
            lab = row["emotions"][i]
            lab_name = LABEL2NAME.get(int(lab), str(lab))
            print(f"{spk} [{lab}:{lab_name}] {dialog[i]}")
        print()


# =========================================================
# Helper: build context-target text for a matched utterance
# Reuses your earlier training/inference format
# =========================================================
def build_context_target_text(dialog: List[str], target_idx: int) -> str:
    context_lines = []
    for i, utt in enumerate(dialog[:target_idx]):
        speaker = "A" if i % 2 == 0 else "B"
        context_lines.append(f"{speaker}: {utt}")

    context = "\n".join(context_lines) if context_lines else "[No prior context]"
    target_speaker = "A" if target_idx % 2 == 0 else "B"
    target = dialog[target_idx]

    text = (
        f"Context:\n{context}\n\n"
        f"Target utterance ({target_speaker}): {target}"
    )
    return text


# =========================================================
# Optional: convert shift matches into model-ready inference inputs
# =========================================================
def shifts_to_inference_df(shifts_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in shifts_df.iterrows():
        dialog = row["dialog"]
        target_idx = row["curr_global_turn_idx"]

        rows.append({
            "dialogue_row_idx": row["dialogue_row_idx"],
            "speaker": row["speaker"],
            "old_label": row["old_label"],
            "old_label_name": row["old_label_name"],
            "new_label": row["new_label"],
            "new_label_name": row["new_label_name"],
            "target_idx": target_idx,
            "model_input_text": build_context_target_text(dialog, target_idx),
            "target_utterance": dialog[target_idx],
        })

    return pd.DataFrame(rows)


# =========================================================
# Example usage
# =========================================================
# Case 1: if you already have a dataframe with columns:
#   df["dialog"], df["emotion"]
#
# shifts_df = find_speaker_label_shifts(
#     df,
#     dialog_col="dialog",
#     emotion_col="emotion",
#     require_nonzero=True,
#     direct_consecutive_for_speaker=True,
# )
#
# print("Total shift matches:", len(shifts_df))
# print_shift_examples(shifts_df, n=5)
#
# Example: only happiness -> anger
# happy_to_angry = filter_specific_shift(shifts_df, from_label=4, to_label=1)
# print("Happiness -> anger:", len(happy_to_angry))
# print_shift_examples(happy_to_angry, n=5)
#
# Example: prepare those matched cases for your classifier
# inf_df = shifts_to_inference_df(happy_to_angry)
# print(inf_df.head(3)[["speaker", "old_label_name", "new_label_name", "model_input_text"]])

In [4]:
shifts_df = find_speaker_label_shifts(
    df,
    dialog_col="dialog",
    emotion_col="emotion",
    require_nonzero=True,
    direct_consecutive_for_speaker=True,
)

In [ ]:
print("Total shift matches:", len(shifts_df))
print_shift_examples(shifts_df, n=5)

Total shift matches: 17
Match #0
Dialogue row idx: 44
Speaker: B
Shift: 5 (sadness) -> 4 (happiness)
Previous turn idx (global): 1
Current  turn idx (global): 3

[Previous utterance]
 Toilets are in the rear , I am afraid all the toilets are fully occupied at the moment . 

[Current utterance]
 You are welcome . 

[Dialogue excerpt up to current turn]
A [0:no emotion] By the way miss , where is the toilet ? 
B [5:sadness]  Toilets are in the rear , I am afraid all the toilets are fully occupied at the moment . 
A [6:surprise]  What ? Oh , what we live ! Thank you very much for your help , miss . 
B [4:happiness]  You are welcome . 

Match #1
Dialogue row idx: 197
Speaker: B
Shift: 2 (disgust) -> 1 (anger)
Previous turn idx (global): 1
Current  turn idx (global): 3

[Previous utterance]
 Coffee ? I don ’ t honestly like that kind of stuff . 

[Current utterance]
 What ’ s wrong with that ? Cigarette is the thing I go crazy for . 

[Dialogue excerpt up to current turn]
A [4:happiness] So

### HF

In [3]:
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

In [4]:
DAILYDIALOG_ID2EMOTION = {
    0: "neutral",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise",
}

BASIC_EMOTION_TO_VAD = {
    "neutral":   (0.50, 0.20, 0.50),
    "anger":     (0.20, 0.80, 0.65),
    "disgust":   (0.15, 0.60, 0.45),
    "fear":      (0.10, 0.85, 0.20),
    "happiness": (0.85, 0.65, 0.70),
    "sadness":   (0.15, 0.25, 0.25),
    "surprise":  (0.55, 0.85, 0.45),
}

In [5]:
def try_load_dataset(specs: List[Tuple[Any, ...]]):
    errors = []

    for spec in specs:
        try:
            return load_dataset(*spec)
        except Exception as e:
            errors.append(f"{spec}: {repr(e)}")

    raise RuntimeError("Could not load dataset. Errors:\n" + "\n".join(errors))

In [6]:

def clean_text(text: Any) -> Optional[str]:
    if not isinstance(text, str):
        return None

    text = text.replace("_comma_", ",")
    text = re.sub(r"\s+", " ", text).strip()

    # Remove simple speaker prefixes if your synthetic data has them.
    text = re.sub(r"^(speaker\s*)?[AB]\s*:\s*", "", text, flags=re.IGNORECASE)

    if not text:
        return None

    return text

In [7]:
def first_existing(example: Dict[str, Any], keys: List[str]) -> Any:
    for key in keys:
        if key in example and example[key] is not None:
            return example[key]
    return None

In [14]:
def standardize_split(split: Any) -> str:
    s = str(split).lower().strip()

    if s in {"valid", "validation", "dev"}:
        return "validation"
    if s in {"test", "testing"}:
        return "test"
    return "train"

In [8]:
def make_row(
    text: str,
    vad: Tuple[float, float, float],
    source: str,
    split: str,
    label_source: str,
    original_label: Optional[Any] = None,
) -> Dict[str, Any]:
    v, a, d = vad

    return {
        "text": text,
        "labels": [float(v), float(a), float(d)],
        "valence": float(v),
        "arousal": float(a),
        "dominance": float(d),
        "source": source,
        "split": standardize_split(split),
        "label_source": label_source,
        "original_label": None if original_label is None else str(original_label),
    }

In [9]:
def load_dailydialog_as_vad() -> pd.DataFrame:
    ds = try_load_dataset([
        ("roskoN/dailydialog", "full"),
        ("roskoN/dailydialog",),
        ("daily_dialog",),
    ])

    rows = []

    for split_name, split_ds in ds.items():
        for example in split_ds:
            utterances = first_existing(example, ["utterances", "dialog", "dialogue"])
            emotions = first_existing(example, ["emotions", "emotion"])

            if utterances is None or emotions is None:
                continue

            for utterance, emotion_id in zip(utterances, emotions):
                text = clean_text(utterance)
                if text is None:
                    continue

                try:
                    emotion_id = int(emotion_id)
                except Exception:
                    continue

                emotion_name = DAILYDIALOG_ID2EMOTION.get(emotion_id, "neutral")
                vad = BASIC_EMOTION_TO_VAD[emotion_name]

                rows.append(
                    make_row(
                        text=text,
                        vad=vad,
                        source="DailyDialog",
                        split=split_name,
                        label_source="categorical_to_vad_prototype",
                        original_label=emotion_name,
                    )
                )

    return pd.DataFrame(rows)


In [15]:
df = load_dailydialog_as_vad()

In [29]:
df

,text,labels,valence,arousal,dominance,source,split,label_source,original_label
0,"Say , Jim , how about going for a few beers af...","[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,train,categorical_to_vad_prototype,neutral
1,You know that is tempting but is really not go...,"[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,train,categorical_to_vad_prototype,neutral
2,What do you mean ? It will help us to relax .,"[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,train,categorical_to_vad_prototype,neutral
3,Do you really think so ? I don't . It will jus...,"[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,train,categorical_to_vad_prototype,neutral
4,I guess you are right.But what shall we do ? I...,"[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,train,categorical_to_vad_prototype,neutral
...,...,...,...,...,...,...,...,...,...
102974,are you kidding ? Can you afford it ? Do you t...,"[0.55, 0.85, 0.45]",0.55,0.85,0.45,DailyDialog,test,categorical_to_vad_prototype,surprise
102975,"never mind that , I'll take care of it . Are y...","[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,test,categorical_to_vad_prototype,neutral
102976,"yeah , I think so .","[0.5, 0.2, 0.5]",0.50,0.20,0.50,DailyDialog,test,categorical_to_vad_prototype,neutral
102977,ok . I'll make the arrangements . It will be g...,"[0.85, 0.65, 0.7]",0.85,0.65,0.70,DailyDialog,test,categorical_to_vad_prototype,happiness


In [31]:
for i in range(10):
    print(df['text'].iloc[i+10] + "   " + df['labels'].iloc[i+10].__str__())

Can you do push-ups ?   [0.5, 0.2, 0.5]
Of course I can . It's a piece of cake ! Believe it or not , I can do 30 push-ups a minute .   [0.5, 0.2, 0.5]
Really ? I think that's impossible !   [0.55, 0.85, 0.45]
You mean 30 push-ups ?   [0.5, 0.2, 0.5]
Yeah !   [0.5, 0.2, 0.5]
It's easy . If you do exercise everyday , you can make it , too .   [0.5, 0.2, 0.5]
Can you study with the radio on ?   [0.5, 0.2, 0.5]
No , I listen to background music .   [0.5, 0.2, 0.5]
What is the difference ?   [0.5, 0.2, 0.5]
The radio has too many comerials .   [0.5, 0.2, 0.5]


In [32]:
import transformers
from transformers import TrainingArguments
import inspect

print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
print("TrainingArguments from:", inspect.getfile(TrainingArguments))
print(inspect.signature(TrainingArguments.__init__))

transformers version: 5.7.0
transformers file: c:\Users\Michael Lin\projects\.venv\Lib\site-packages\transformers\__init__.py
TrainingArguments from: c:\Users\Michael Lin\projects\.venv\Lib\site-packages\transformers\training_args.py
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_fu

## EmpDynamic

In [1]:
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset


In [2]:
def read_json_or_jsonl(path: Path) -> List[Any]:
    if path.suffix.lower() == ".jsonl":
        records = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records

    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, list):
        return obj

    return [obj]

In [ ]:
def load_emobank_as_vad(emobank_path: Optional[str]) -> pd.DataFrame:
    """
    EmoBank usually has columns like:
    id, split, V, A, D, text

    The V/A/D values are commonly on a 1-5 scale, so this function
    normalizes them into [0, 1].
    """
    if emobank_path is None:
        emobank_path = (
            "EmoBank\corpus\emobank.csv"
        )

    df = pd.read_csv(emobank_path)

    text_col = find_column(df, ["text", "sentence", "utterance"])
    v_col = find_column(df, ["V", "valence", "val"])
    a_col = find_column(df, ["A", "arousal", "aro"])
    d_col = find_column(df, ["D", "dominance", "dom"])
    split_col = find_column(df, ["split", "partition"])

    required = {
        "text": text_col,
        "valence": v_col,
        "arousal": a_col,
        "dominance": d_col,
    }

    missing = [name for name, col in required.items() if col is None]
    if missing:
        raise ValueError(
            f"Missing required EmoBank columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    rows = []

    for _, row in df.iterrows():
        text = clean_text(row[text_col])
        if text is None:
            continue

        split = row[split_col] if split_col is not None else "train"

        vad = normalize_vad_triplet(
            row[v_col],
            row[a_col],
            row[d_col],
            scale="auto",
        )

        rows.append(
            make_row(
                text=text,
                vad=vad,
                source="EmoBank",
                split=split,
                label_source="gold_vad",
                original_label=None,
            )
        )

    return pd.DataFrame(rows)

## Qwen

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

#"C:\\Users\\Michael Lin\\projects\\IOAI2025\\qwen3-8b-local"

model_name = "C://Users//Michael Lin//projects//Models//models--Qwen--Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Expand the following sentence: 'The cat is on the mat.'"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# qwen3_dailydialog_test.py
import json
from pathlib import Path
from typing import List

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import random

In [ ]:
# =========================
# Config
# =========================
MODEL_NAME = "C:\\Users\\Michael Lin\\projects\\IOAI2025\\qwen3-8b-local"
MAX_NEW_TOKENS = 8
OUTPUT_FILE = "qwen3_dailydialog_emotion_predictions2.jsonl"

LABEL2NAME = {
    0: "no emotion",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise",
}
VALID_LABELS = set(LABEL2NAME.keys())

In [ ]:
# =========================
# Parsing helpers
# =========================

def extract_first_valid_label(text):
    """
    Robustly extract the first integer 0-6 from model output.
    """
    text = text.strip()
    m = re.search(r"\b([0-6])\b", text)
    if m:
        return int(m.group(1))
    return None


# =========================
# Prompt builder
# =========================
def build_messages_for_index(dialog, target_idx, include_target_in_dialogue=True):
    """
    Build a prompt to classify the emotion of dialog[target_idx].

    include_target_in_dialogue=True:
        The dialogue shown includes the target utterance, marked as target.

    include_target_in_dialogue=False:
        The dialogue shown contains only prior context, and the target utterance
        is shown separately below.
    """
    if include_target_in_dialogue:
        lines = []
        for i, utt in enumerate(dialog[:target_idx + 1]):
            speaker = "A" if i % 2 == 0 else "B"
            marker = " <-- target utterance" if i == target_idx else ""
            lines.append(f"{speaker}: {utt}{marker}")
        conversation = "\n".join(lines)
    else:
        lines = []
        for i, utt in enumerate(dialog[:target_idx]):
            speaker = "A" if i % 2 == 0 else "B"
            lines.append(f"{speaker}: {utt}")
        conversation = "\n".join(lines) if lines else "[No prior context]"

    target_utt = dialog[target_idx]

    label_desc = (
        "0 = no emotion\n"
        "1 = anger\n"
        "2 = disgust\n"
        "3 = fear\n"
        "4 = happiness\n"
        "5 = sadness\n"
        "6 = surprise"
    )

    user_prompt = f"""
You are doing emotion classification on DailyDialog.

Classify the emotion of the target utterance.
Return ONLY one integer from 0 to 6.
DO NOT explain your answer. DO NOT return anything other than the integer.
DO NOT OUTPUT ANYTHING ELSE.

Label mapping:
{label_desc}

Context dialogue:
{conversation}

Target utterance:
{target_utt}

Answer:
""".strip()

    return [
        {"role": "system", "content": "You are a precise text classification assistant."},
        {"role": "user", "content": user_prompt},
    ]



def build_eval_instances(
    df,
    zero_ratio=0.25,
    seed=42,
    dialog_col_idx=0,
    emotion_col_idx=2,
):
    """
    Build utterance-level evaluation instances.

    Rules:
    - keep every utterance with non-zero label
    - keep only a proportion of zero-label utterances

    zero_ratio:
        proportion of zero-label utterances to keep globally, e.g.
        0.25 means keep 25% of all zero-labeled utterances
    """
    rng = random.Random(seed)

    nonzero_instances = []
    zero_instances = []
    skipped_rows = 0
    bad_pairs = 0

    for ex in df.itertuples(index=False):
        dialog = parse_dialog_text(ex[dialog_col_idx])
        emotions = parse_label_list(ex[emotion_col_idx])

        if dialog is None or emotions is None:
            skipped_rows += 1
            continue

        if len(dialog) != len(emotions):
            bad_pairs += 1
            continue

        for idx, label in enumerate(emotions):
            item = {
                "dialog": dialog,
                "target_idx": idx,
                "true_label": int(label),
            }

            if int(label) == 0:
                zero_instances.append(item)
            else:
                nonzero_instances.append(item)

    if zero_ratio < 0:
        zero_ratio = 0.0
    if zero_ratio > 1:
        zero_ratio = 1.0

    keep_zero_n = int(len(zero_instances) * zero_ratio)
    sampled_zero_instances = rng.sample(zero_instances, keep_zero_n) if keep_zero_n < len(zero_instances) else zero_instances

    instances = nonzero_instances + sampled_zero_instances
    rng.shuffle(instances)

    stats = {
        "skipped_rows": skipped_rows,
        "bad_pairs": bad_pairs,
        "nonzero_instances": len(nonzero_instances),
        "zero_instances_total": len(zero_instances),
        "zero_instances_kept": len(sampled_zero_instances),
        "total_instances": len(instances),
    }

    return instances, stats

In [ ]:
out = parse_label_list(df['emotion'].iloc[0])
out

In [ ]:


def main(
    df,
    zero_ratio=0.25,
    seed=42,
    include_target_in_dialogue=True,
    max_instances=None,
):
    
    ind = 0

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        device_map=0,
        trust_remote_code=True,
    )
    model.eval()

    instances, stats = build_eval_instances(
        df=df,
        zero_ratio=zero_ratio,
        seed=seed,
        dialog_col_idx=0,
        emotion_col_idx=2,
    )

    print("\nInstance construction summary:")
    for k, v in stats.items():
        print(f"{k}: {v}")

    if max_instances is not None:
        instances = instances[:max_instances]
        print(f"Using first {len(instances)} instances after sampling/shuffle.")

    y_true = []
    y_pred = []
    unparsable_outputs = 0

    out_path = Path(OUTPUT_FILE)
    with out_path.open("w", encoding="utf-8") as f:
        for item in tqdm(instances, desc="Running inference"):
            dialog = item["dialog"]
            target_idx = item["target_idx"]
            true_label = item["true_label"]

            messages = build_messages_for_index(
                dialog=dialog,
                target_idx=target_idx,
                include_target_in_dialogue=include_target_in_dialogue,
            )

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                enable_thinking=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(text, return_tensors="pt").to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

            gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
            #print(outputs[0])
            #print(outputs)
            raw_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            #print(raw_output)
            pred_label = extract_first_valid_label(raw_output)

            record = {
                "target_idx": target_idx,
                "target_utterance": dialog[target_idx],
                "context": dialog[:target_idx],
                "dialog_up_to_target": dialog[:target_idx + 1],
                "true_label": true_label,
                "true_name": LABEL2NAME[true_label],
                "raw_output": raw_output,
                "pred_label": pred_label,
                "pred_name": LABEL2NAME[pred_label] if pred_label in LABEL2NAME else None,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            if pred_label in VALID_LABELS:
                y_true.append(true_label)
                y_pred.append(pred_label)
            else:
                unparsable_outputs += 1

    print("\nFinished.")
    print(f"Evaluated instances: {len(instances)}")
    print(f"Valid predictions:   {len(y_true)}")
    print(f"Unparsable outputs:  {unparsable_outputs}")
    print(f"Saved outputs to:    {out_path.resolve()}")

    if len(y_true) > 0:
        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        weighted_f1 = f1_score(y_true, y_pred, average="weighted")

        print(f"\nAccuracy:    {acc:.4f}")
        print(f"Macro-F1:    {macro_f1:.4f}")
        print(f"Weighted-F1: {weighted_f1:.4f}")

        print("\nClassification report:")
        print(classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3, 4, 5, 6],
            target_names=[LABEL2NAME[i] for i in range(7)],
            digits=4
        ))

        print("\nConfusion matrix:")
        print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4, 5, 6]))

In [ ]:
main(df)

In [ ]:


s = "['Hey man , you wanna buy some weed ? ' ' Some what ? '\n ' Weed ! You know ? Pot , Ganja , Mary Jane some chronic ! '\n ' Oh , umm , no thanks . '\n ' I also have blow if you prefer to do a few lines . '\n ' No , I am ok , really . '\n ' Come on man ! I even got dope and acid ! Try some ! '\n ' Do you really have all of these drugs ? Where do you get them from ? '\n ' I got my connections ! Just tell me what you want and I ’ ll even give you one ounce for free . '\n ' Sounds good ! Let ’ s see , I want . ' ' Yeah ? '\n ' I want you to put your hands behind your head ! You are under arrest ! ']"

result = re.findall(r"'(.*?)'", s)

print(result)

In [ ]:
import pandas as pd

file_name = "test"

df = pd.read_csv(f"DailyDialogue/{file_name}.csv")

## RoBERTa

In [1]:
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Optional

import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from collections import Counter

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

# =========================================================
# Reuse these from last time
# =========================================================
# def parse_dialog_text(x): ...
# def parse_label_list(x): ...
#
# Assumed behavior:
# - parse_dialog_text(x) -> list[str] or None
# - parse_label_list(x) -> list[int] or None


# =========================================================
# Config
# =========================================================
MODEL_NAME = "roberta-base"
# MODEL_NAME = "microsoft/deberta-v3-base"

NUM_LABELS = 7
MAX_LENGTH = None
ZERO_RATIO_TRAIN = 0.25
ZERO_RATIO_EVAL = 0.25
SEED = 42

LABEL2NAME = {
    0: "no emotion",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise",
}


# =========================================================
# Example building
# =========================================================
def build_context_target_text(dialog: List[str], target_idx: int) -> str:
    """
    Build one classification text:
    prior conversation as context + current utterance as target.
    """
    context_lines = []
    for i, utt in enumerate(dialog[:target_idx]):
        speaker = "A" if i % 2 == 0 else "B"
        context_lines.append(f"{speaker}: {utt}")

    context = "\n".join(context_lines) if context_lines else "[No prior context]"
    target_speaker = "A" if target_idx % 2 == 0 else "B"
    target = dialog[target_idx]

    text = (
        f"Context:\n{context}\n\n"
        f"Target utterance ({target_speaker}): {target}"
    )
    return text


def build_utterance_examples_from_df(
    df: pd.DataFrame,
    zero_ratio: float = 0.25,
    seed: int = 42,
    dialog_col_idx: int = 0,
    emotion_col_idx: int = 2,
) -> List[Dict]:
    """
    Build utterance-level examples from a dataframe.

    Keeps:
    - all non-zero emotion utterances
    - a sampled proportion of zero emotion utterances
    """
    rng = random.Random(seed)

    zero_pool = []
    nonzero_pool = []

    skipped_rows = 0
    mismatched_rows = 0

    for row in df.itertuples(index=False):
        dialog = parse_dialog_text(row[dialog_col_idx])
        emotions = parse_label_list(row[emotion_col_idx])

        if dialog is None or emotions is None:
            skipped_rows += 1
            continue

        if len(dialog) != len(emotions):
            mismatched_rows += 1
            continue

        for idx, label in enumerate(emotions):
            ex = {
                "text": build_context_target_text(dialog, idx),
                "labels": int(label),
                "target_idx": idx,
                "dialog_len": len(dialog),
                "target_utterance": dialog[idx],
            }
            if int(label) == 0:
                zero_pool.append(ex)
            else:
                nonzero_pool.append(ex)

    keep_zero_n = int(len(zero_pool) * max(0.0, min(1.0, zero_ratio)))
    zero_kept = rng.sample(zero_pool, keep_zero_n) if keep_zero_n < len(zero_pool) else zero_pool

    examples = nonzero_pool + zero_kept
    rng.shuffle(examples)

    print("\nExample construction summary")
    print(f"Skipped rows:        {skipped_rows}")
    print(f"Mismatched rows:     {mismatched_rows}")
    print(f"Non-zero examples:   {len(nonzero_pool)}")
    print(f"Zero examples total: {len(zero_pool)}")
    print(f"Zero examples kept:  {len(zero_kept)}")
    print(f"Total examples:      {len(examples)}")

    return examples


# =========================================================
# Tokenization
# =========================================================
def tokenize_function(batch, tokenizer):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


# =========================================================
# Metrics
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


# =========================================================
# Main train/eval pipeline
# =========================================================
def run_experiment(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str = MODEL_NAME,
    output_dir: str = "./emotion_cls_output",
):
    set_seed(SEED)

    print(f"\nLoading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print("Building train examples...")
    train_examples = build_utterance_examples_from_df(
        train_df,
        zero_ratio=ZERO_RATIO_TRAIN,
        seed=SEED,
    )

    print("\nBuilding validation examples...")
    valid_examples = build_utterance_examples_from_df(
        valid_df,
        zero_ratio=ZERO_RATIO_EVAL,
        seed=SEED,
    )

    print("\nBuilding test examples...")
    test_examples = build_utterance_examples_from_df(
        test_df,
        zero_ratio=ZERO_RATIO_EVAL,
        seed=SEED,
    )

    train_ds = Dataset.from_list(train_examples)
    valid_ds = Dataset.from_list(valid_examples)
    test_ds = Dataset.from_list(test_examples)

    print("\nTokenizing datasets...")
    train_ds = train_ds.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    valid_ds = valid_ds.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    test_ds = test_ds.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    keep_cols = {"input_ids", "attention_mask", "labels"}
    if "token_type_ids" in train_ds.column_names:
        keep_cols.add("token_type_ids")

    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep_cols])
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols])

    print(f"\nLoading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label={i: LABEL2NAME[i] for i in range(NUM_LABELS)},
        label2id={LABEL2NAME[i]: i for i in range(NUM_LABELS)},
    )


    print("Train label counts:", Counter([x["labels"] for x in train_examples]))
    print("Valid label counts:", Counter([x["labels"] for x in valid_examples]))
    print("Test  label counts:", Counter([x["labels"] for x in test_examples]))

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=20,
        logging_first_step=True,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=6,
        learning_rate=1e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        label_names=["labels"],
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("\nStarting training...")
    trainer.train()

    print("\nValidation evaluation...")
    valid_metrics = trainer.evaluate(valid_ds)
    print(valid_metrics)

    print("\nTest evaluation...")
    test_metrics = trainer.evaluate(test_ds)
    print(test_metrics)

    print("\nGenerating detailed test report...")
    pred_output = trainer.predict(test_ds)
    test_logits = pred_output.predictions
    test_labels = pred_output.label_ids
    test_preds = np.argmax(test_logits, axis=-1)

    print("\nClassification report:")
    print(classification_report(
        test_labels,
        test_preds,
        labels=list(range(NUM_LABELS)),
        target_names=[LABEL2NAME[i] for i in range(NUM_LABELS)],
        digits=4,
        zero_division=0,
    ))

    print("\nConfusion matrix:")
    print(confusion_matrix(
        test_labels,
        test_preds,
        labels=list(range(NUM_LABELS)),
    ))

    return trainer, valid_metrics, test_metrics


# =========================================================
# Example usage
# =========================================================
# Assume you already have three dataframes:
# train_df, valid_df, test_df
#
# And each row roughly contains:
# - column 0: dialog
# - column 2: emotion
#
# Example:
# trainer, valid_metrics, test_metrics = run_experiment(
#     train_df=train_df,
#     valid_df=valid_df,
#     test_df=test_df,
#     model_name="roberta-base",
#     output_dir="./roberta_dailydialog_emotion"
# )
#
# trainer, valid_metrics, test_metrics = run_experiment(
#     train_df=train_df,
#     valid_df=valid_df,
#     test_df=test_df,
#     model_name="microsoft/deberta-v3-base",
#     output_dir="./deberta_dailydialog_emotion"
# )

In [ ]:
test_df = pd.read_csv("DailyDialogue/test.csv")
train_df = pd.read_csv("DailyDialogue/train.csv")
valid_df = pd.read_csv("DailyDialogue/validation.csv")

In [ ]:
# =========================================================
# Example usage
# =========================================================
# Assume you already have three dataframes:
# train_df, valid_df, test_df
#
# And each row roughly contains:
# - column 0: dialog
# - column 2: emotion
#
# Example:
# trainer, valid_metrics, test_metrics = run_experiment(
#     train_df=train_df,
#     valid_df=valid_df,
#     test_df=test_df,
#     model_name="microsoft/deberta-v3-base",
#     output_dir="./deberta_dailydialog"
# )
#
trainer, valid_metrics, test_metrics = run_experiment(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    model_name="roberta-base",
    output_dir="./roberta_dailydialog"
)

### Inference

In [10]:
import re

def raw_dialogue_to_list(raw_text: str):
    """
    Split a raw multi-turn dialogue block into a list of utterances.
    Assumes each utterance is separated by one or more blank lines.
    """
    raw_text = raw_text.strip()

    # split on blank lines
    parts = re.split(r"\n\s*\n", raw_text)

    # normalize whitespace inside each utterance
    parts = [" ".join(p.strip().split()) for p in parts if p.strip()]

    return parts

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# Config
# =========================
CHECKPOINT_PATH = "./roberta_dailydialog/checkpoint-13055"  
# or the final saved model folder

LABEL2NAME = {
    0: "no emotion",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise",
}

MAX_LENGTH = 256

# =========================
# Load model
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_PATH)
model.to(device)
model.eval()




# =========================
# Inference function
# =========================
def predict_emotion(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())

    return {
        "text": text,
        "pred_label_id": pred_id,
        "pred_label_name": LABEL2NAME[pred_id],
        "probabilities": {
            LABEL2NAME[i]: float(probs[i].item()) for i in range(len(LABEL2NAME))
        }
    }




Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [45]:
text = """

No... Of course we understand. Sorry for getting too excited. its just... maybe you should talk through with us sometimes. Mom's there. I'm here, and you can talk to us when you felt necessary
"""
a = raw_dialogue_to_list(text)

In [46]:
b = build_context_target_text(a, 0)

In [49]:
type(b)

str

In [50]:
result = predict_emotion(b)

print("Predicted label:", result["pred_label_id"], result["pred_label_name"])
print("Probabilities:")
for k, v in result["probabilities"].items():
    print(f"{k:12s}: {v:.4f}")

Predicted label: 0 no emotion
Probabilities:
no emotion  : 0.9995
anger       : 0.0000
disgust     : 0.0000
fear        : 0.0000
happiness   : 0.0003
sadness     : 0.0001
surprise    : 0.0000


# Encoder Model

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple
from collections import defaultdict
import math

import torch
from torch import Tensor, nn
from torch.utils.data import Dataset, DataLoader

In [ ]:

def normalize_vad_1to01(vad: Tensor) -> Tensor:
    """
    EmoBank-style [-1, 1] VAD -> [0, 1].
    """
    return (vad + 1.0) / 2.0

In [ ]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int = 2,
        dropout: float = 0.1,
        activation: type[nn.Module] = nn.GELU,
    ):
        super().__init__()

        layers: List[nn.Module] = []
        d = in_dim

        for _ in range(max(depth - 1, 1)):
            layers += [
                nn.Linear(d, hidden_dim),
                activation(),
                nn.Dropout(dropout),
            ]
            d = hidden_dim

        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [ ]:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F


    class BasicResidualVADGRU(nn.Module):
        """
        Minimal personality-conditioned residual GRU.

        Predicts:

            z_A_hat(t) = z_A(t-1) + delta_z(t)

        where:

            delta_z(t) = GRU(z_A(t-1), z_B(t), z_B(t)-z_A(t-1), personality_A)

        Expected VAD range: [-1, 1]
        Expected personality range: usually [0, 1] for Big Five.
        """

        def __init__(
            self,
            vad_dim=3,
            personality_dim=5,
            hidden_dim=64,
            max_delta=0.5,
        ):
            super().__init__()

            self.vad_dim = vad_dim
            self.personality_dim = personality_dim
            self.hidden_dim = hidden_dim
            self.max_delta = max_delta

            input_dim = vad_dim + vad_dim + vad_dim + personality_dim
            #           z_A_prev + z_B_now + difference + personality

            self.gru = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                batch_first=True,
            )

            self.delta_head = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, vad_dim),
            )

        def forward(self, z_self_prev, z_other, personality, h0=None):
            """
            Args:
                z_self_prev:
                    Tensor [B, T, 3]
                    Previous own VAD, z_A(t-1)

                z_other:
                    Tensor [B, T, 3]
                    Counterpart perceived/current VAD, z_B(t)

                personality:
                    Tensor [B, 5]
                    Big Five vector for speaker A

                h0:
                    Optional initial hidden state [1, B, hidden_dim]

            Returns:
                z_pred:
                    Tensor [B, T, 3]

                delta:
                    Tensor [B, T, 3]

                h:
                    Final GRU hidden state
            """

            B, T, _ = z_self_prev.shape

            personality_seq = personality.unsqueeze(1).expand(B, T, -1)

            diff = z_other - z_self_prev

            x = torch.cat(
                [
                    z_self_prev,
                    z_other,
                    diff,
                    personality_seq,
                ],
                dim=-1,
            )

            gru_out, h = self.gru(x, h0)

            raw_delta = self.delta_head(gru_out)

            # Bound the per-turn shift.
            delta = self.max_delta * torch.tanh(raw_delta)

            z_pred = z_self_prev + delta

            # Keep normalized VAD inside valid range.
            z_pred = torch.clamp(z_pred, -1.0, 1.0)

            return z_pred, delta, h

In [ ]:
def masked_mse(pred, target, mask=None):
    """
    pred:   [B, T, 3]
    target: [B, T, 3]
    mask:   [B, T], optional
    """

    loss = ((pred - target) ** 2).mean(dim=-1)

    if mask is not None:
        loss = loss * mask
        return loss.sum() / mask.sum().clamp_min(1.0)

    return loss.mean()

In [ ]:
def masked_residual_mse(pred, target, z_prev, mask=None):
    """
    Compares predicted emotional shift against true emotional shift.
    """

    pred_delta = pred - z_prev
    true_delta = target - z_prev

    loss = ((pred_delta - true_delta) ** 2).mean(dim=-1)

    if mask is not None:
        loss = loss * mask
        return loss.sum() / mask.sum().clamp_min(1.0)

    return loss.mean()

In [ ]:
def train_step(model, batch, optimizer, device="cuda"):
    model.train()

    z_self_prev = batch["z_self_prev"].to(device)   # [B, T, 3]
    z_other = batch["z_other"].to(device)           # [B, T, 3]
    z_target = batch["z_target"].to(device)         # [B, T, 3]
    personality = batch["personality"].to(device)   # [B, 5]
    mask = batch.get("mask", None)

    if mask is not None:
        mask = mask.to(device)

    z_pred, delta, h = model(
        z_self_prev=z_self_prev,
        z_other=z_other,
        personality=personality,
    )

    loss_state = masked_mse(z_pred, z_target, mask)
    loss_delta = masked_residual_mse(
        z_pred,
        z_target,
        z_self_prev,
        mask,
    )

    loss = loss_state + 0.5 * loss_delta

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return {
        "loss": loss.item(),
        "loss_state": loss_state.item(),
        "loss_delta": loss_delta.item(),
    }

# Random

In [1]:
import torch

ckpt_path = "emotion_gru_runs/run1/best_model.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

print(type(ckpt))

if isinstance(ckpt, dict):
    print("Checkpoint keys:")
    for k in ckpt.keys():
        print(" ", k)

    # Try to find model state dict
    state = (
        ckpt.get("model_state_dict")
        or ckpt.get("state_dict")
        or ckpt.get("model")
        or ckpt
    )

    print("\nState dict sample:")
    if isinstance(state, dict):
        for i, (k, v) in enumerate(state.items()):
            print(k, tuple(v.shape) if hasattr(v, "shape") else type(v))
            if i >= 20:
                break
else:
    print("This may be a full saved model object.")
    print(ckpt)

<class 'dict'>
Checkpoint keys:
  model_state_dict
  optimizer_state_dict
  epoch
  best_val_mse
  config
  big5_keys
  vad_keys

State dict sample:
input_proj.0.weight (64, 14)
input_proj.0.bias (64,)
input_proj.1.weight (64,)
input_proj.1.bias (64,)
input_proj.4.weight (64, 64)
input_proj.4.bias (64,)
init_state.0.weight (64, 8)
init_state.0.bias (64,)
gru.weight_ih_l0 (192, 64)
gru.weight_hh_l0 (192, 64)
gru.bias_ih_l0 (192,)
gru.bias_hh_l0 (192,)
vad_decoder.0.weight (64, 64)
vad_decoder.0.bias (64,)
vad_decoder.3.weight (3, 64)
vad_decoder.3.bias (3,)


In [1]:
a = [1,2,3,4,5,6,7,8,9,10]
a[4:]

[5, 6, 7, 8, 9, 10]